# 15. Korean morphology retrieval ablation

이 노트북은 **Codex coder agent가 `skn25` 환경에서 실행한 개발셋 전용 실험**이다. 기존 사용자 실행 노트북의 provenance와 구분한다. 개발 질의 30개만 사용하며 독립 holdout은 읽거나 평가하지 않는다. 결과는 후보 선정 근거일 뿐 운영 일반화 결론이 아니다.

사전 고정한 판정 규칙은 다음과 같다. Primary는 evidence-only RRF MRR@5 delta `>= +0.025`이다. Guardrail은 evidence Strict Hit@3, Recall@5, nDCG@5, evidence Card Hit@3, card-group Card Hit@3의 모두 비회귀다. 모두 만족할 때만 `promote_morphology_to_holdout_candidate`, 아니면 `do_not_promote_morphology`다. `0 < delta < .025`는 inconclusive/non-promotion이며 우월로 표현하지 않는다. BM25만 개선되고 RRF가 기준 미달이어도 비채택한다.

실행은 `conda run -n skn25` 아래 `nbclient 0.10.4`의 `NotebookClient(timeout=900, kernel_name='python3')`로 in-place 수행했다. 실행 드라이버는 노트북 저장 후 manifest의 노트북 raw SHA-256만 최종화한다.

In [1]:
from __future__ import annotations

import csv
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import sqlite3
import tempfile
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import chromadb
import kiwipiepy
import numpy as np
from chromadb.config import Settings
from kiwipiepy import Kiwi

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl').is_file())
SOURCE_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
OUTPUT_ROOT = PROJECT_ROOT / 'notebooks/data/15_korean_morphology_retrieval_ablation'
NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks/15_korean_morphology_retrieval_ablation.ipynb'
CHROMA_ROOT = SOURCE_ROOT / 'chroma'
EMBEDDING_MODEL = 'text-embedding-3-small'
RAW_TOKEN = re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*', re.IGNORECASE)
POS_TAGS = ('NNG', 'NNP', 'VV', 'VA')
BM25_K1, BM25_B = 1.5, 0.75
RRF_K, RRF_DEPTH = 60, 50
RRF_WEIGHTS = (0.5, 0.5)
CONFIGURATION_NAMES = (
    'normalized_bm25',
    'normalized_plus_morph_bm25',
    'rrf_vector_normalized',
    'rrf_vector_normalized_plus_morph',
)
PROMOTION_RULE = {
    'primary': {'group': 'evidence', 'metric': 'mrr_at_5', 'minimum_delta': 0.025},
    'guardrails': [
        {'group': 'evidence', 'metric': 'strict_evidence_hit_at_3'},
        {'group': 'evidence', 'metric': 'recall_at_5'},
        {'group': 'evidence', 'metric': 'ndcg_at_5'},
        {'group': 'evidence', 'metric': 'card_hit_at_3'},
        {'group': 'card', 'metric': 'card_hit_at_3'},
    ],
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'agent_provenance': 'Codex coder agent', 'environment': 'skn25', 'configurations': CONFIGURATION_NAMES, 'promotion_rule': PROMOTION_RULE})

{'agent_provenance': 'Codex coder agent', 'environment': 'skn25', 'configurations': ('normalized_bm25', 'normalized_plus_morph_bm25', 'rrf_vector_normalized', 'rrf_vector_normalized_plus_morph'), 'promotion_rule': {'primary': {'group': 'evidence', 'metric': 'mrr_at_5', 'minimum_delta': 0.025}, 'guardrails': [{'group': 'evidence', 'metric': 'strict_evidence_hit_at_3'}, {'group': 'evidence', 'metric': 'recall_at_5'}, {'group': 'evidence', 'metric': 'ndcg_at_5'}, {'group': 'evidence', 'metric': 'card_hit_at_3'}, {'group': 'card', 'metric': 'card_hit_at_3'}]}}


## Inputs and immutable-state contract

입력은 13번 실험의 명시된 notebook/artifact, embedding usage에 기록된 정확한 6개 cache fingerprint, SQLite read-only 정보로 제한한다. 원본 Chroma에는 `PersistentClient`를 연결하지 않는다. NumPy 완전 순위를 먼저 시도하고 published vector top-5가 재현되지 않을 때만 Chroma 전체를 `/tmp` byte-copy한 snapshot에 연결한다.

In [2]:
def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def atomic_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_csv(path, rows):
    rows = list(rows)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)


def tree_hash(root):
    digest = hashlib.sha256()
    for directory, _, names in os.walk(root):
        for name in sorted(names):
            path = Path(directory) / name
            digest.update(path.relative_to(root).as_posix().encode())
            digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


def percentile(values, quantile):
    return float(np.percentile(np.asarray(values, dtype=np.float64), quantile))


SOURCE_FILES = {
    'contract_notebook': PROJECT_ROOT / 'notebooks/13_hierarchical_chunking_retrieval.ipynb',
    'chunks': SOURCE_ROOT / 'chunks.jsonl',
    'retrieval_per_query': SOURCE_ROOT / 'retrieval_per_query.csv',
    'normalization_per_query': SOURCE_ROOT / 'retrieval_search_normalization_per_query.csv',
    'normalization_summary': SOURCE_ROOT / 'retrieval_search_normalization_summary.json',
    'embedding_usage': SOURCE_ROOT / 'embedding_usage.json',
    'input_manifest': SOURCE_ROOT / 'input_manifest.json',
    'index_manifest': SOURCE_ROOT / 'index_manifest.json',
    'chroma_sqlite': CHROMA_ROOT / 'chroma.sqlite3',
}
embedding_usage = json.loads(SOURCE_FILES['embedding_usage'].read_text(encoding='utf-8'))
cache_fingerprints = [batch['batch_fingerprint'] for batch in embedding_usage['batches']]
assert cache_fingerprints == [
    '3c81bda6bc8b1e69ca305b0dcc219203baeede6797e91045dbde36244babc3b4',
    '335a583c624bd1fe61aec74282af4e1462af20962c96147ada05ed5a29605fba',
    '93cd3f7a2ca8c9e070421705a575465a92f29966aebae38a583c2da60a6ee97f',
    'b6ae13c54bbba3d250e83c39da8673647b077dfc91f671565b4cef016593fbf2',
    '8433343533d6e8ae60f40c42d7e4a0b43b751bb36e26532e738ff51eac4a9ae0',
    'cb5d9e87e091be498d91fa51d46619ca60c7fe5314b27a6686b017cb8c304e55',
]
CACHE_FILES = {
    f'embedding_cache_{index + 1}': SOURCE_ROOT / 'embedding_cache' / EMBEDDING_MODEL / f'{fingerprint}.npz'
    for index, fingerprint in enumerate(cache_fingerprints)
}
source_hashes_before = {name: sha256_file(path) for name, path in {**SOURCE_FILES, **CACHE_FILES}.items()}
chroma_tree_hash_before = tree_hash(CHROMA_ROOT)

database = SOURCE_FILES['chroma_sqlite'].resolve()
with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
    collection_rows = connection.execute('SELECT id, name, config_json_str FROM collections').fetchall()
    collection_metadata = connection.execute('SELECT key, str_value, int_value, float_value, bool_value FROM collection_metadata').fetchall()
    segment_metadata = connection.execute('SELECT key, str_value, int_value, float_value, bool_value FROM segment_metadata').fetchall()
    chroma_embedding_count_before = connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
assert len(collection_rows) == 1 and chroma_embedding_count_before == 327
collection_id, collection_name, collection_config_raw = collection_rows[0]
collection_config = json.loads(collection_config_raw)
distance_override_rows = [row for row in [*collection_metadata, *segment_metadata] if row[0] in {'hnsw:space', 'space', 'distance'}]
assert collection_config == {} and not distance_override_rows

chunks = [json.loads(line) for line in SOURCE_FILES['chunks'].read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}
baseline_rows = list(csv.DictReader(SOURCE_FILES['retrieval_per_query'].open(encoding='utf-8')))
saved_rows = list(csv.DictReader(SOURCE_FILES['normalization_per_query'].open(encoding='utf-8')))
saved_summary = json.loads(SOURCE_FILES['normalization_summary'].read_text(encoding='utf-8'))
evaluation_by_id = {}
for row in baseline_rows:
    if row['method'] == 'keyword':
        evaluation_by_id[row['query_id']] = {
            'query_id': row['query_id'], 'query': row['query'], 'category': row['category'],
            'expected_card': row['expected_card'], 'expected_level': row['expected_level'],
            'required_terms': json.loads(row['required_terms']),
        }
assert len(chunks) == 327 and len(evaluation_by_id) == 30
search_queries = {query_id: item['query'] for query_id, item in evaluation_by_id.items()}
print({'chunks': len(chunks), 'queries': len(search_queries), 'cache_files': len(CACHE_FILES), 'sqlite_collection_config': collection_config, 'distance_override_rows': distance_override_rows, 'chroma_count': chroma_embedding_count_before})

{'chunks': 327, 'queries': 30, 'cache_files': 6, 'sqlite_collection_config': {}, 'distance_override_rows': [], 'chroma_count': 327}


## Symmetric body/query morphology

Baseline `search_tokens`는 기존 코드를 그대로 복제한다. Kiwi는 `num_workers=1`, bundled default model/default dictionary, user dictionary·동의어·typo correction 없음으로 고정한다. Query와 chunk `document`에만 동일하게 적용하며 metadata/structured/evaluation fields는 ranking input으로 사용하지 않는다. `NNG/NNP/VV/VA`의 analyzer form을 NFKC/lower한 `morph_<tag>_<form>`으로 occurrence마다 하나씩 baseline token 뒤에 추가한다.

In [3]:
normalized_text = lambda value: ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())


def canonical_decimal(value):
    rendered = format(Decimal(str(value).replace(',', '')).normalize(), 'f')
    rendered = rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered
    return '0' if rendered in {'', '-0'} else rendered


def search_tokens(value):
    text = normalized_text(value)
    tokens = list(RAW_TOKEN.findall(text))
    for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?', text):
        joined = run.replace(' ', '')
        for size in (2, 3, 4):
            tokens.extend(f'ko{size}_{joined[index:index + size]}' for index in range(max(0, len(joined) - size + 1)))
    consumed = []
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원', text):
        amount = Decimal(match.group(1).replace(',', '')) * 10000 + Decimal(match.group(2).replace(',', '')) * 1000
        tokens.append(f'money_krw_{canonical_decimal(amount)}')
        consumed.append(match.span())
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원', text):
        if any(left <= match.start() and match.end() <= right for left, right in consumed):
            continue
        multiplier = {'만': 10000, '천': 1000, None: 1}[match.group(2)]
        tokens.append(f"money_krw_{canonical_decimal(Decimal(match.group(1).replace(',', '')) * multiplier)}")
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%', text):
        tokens.append(f'percent_{canonical_decimal(match.group(1))}')
    for match in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)', text):
        prefix = match.group(1) or 'none'
        tokens.append(f'period_{prefix}_{canonical_decimal(match.group(2))}_{match.group(3)}')
    return tokens


kiwi = Kiwi(num_workers=1)


def morphology_tokens(value):
    tokens = []
    for token in kiwi.tokenize(str(value)):
        if token.tag in POS_TAGS:
            form = unicodedata.normalize('NFKC', token.form).lower()
            tokens.append(f'morph_{token.tag.lower()}_{form}')
    return tokens


def search_plus_morph_tokens(value):
    return [*search_tokens(value), *morphology_tokens(value)]


assert canonical_decimal('1') == canonical_decimal('1.0') == '1'
assert 'percent_1' in search_tokens('1.0%')
assert search_plus_morph_tokens('카드 할인')[:len(search_tokens('카드 할인'))] == search_tokens('카드 할인')
assert all(token.startswith(('morph_nng_', 'morph_nnp_', 'morph_vv_', 'morph_va_')) for token in morphology_tokens('카드 혜택을 크게 받는다'))

normalized_documents = {chunk['id']: search_tokens(chunk['document']) for chunk in chunks}
morphology_documents = {chunk['id']: morphology_tokens(chunk['document']) for chunk in chunks}
normalized_plus_morph_documents = {
    identifier: [*normalized_documents[identifier], *morphology_documents[identifier]] for identifier in normalized_documents
}
normalized_queries = {query_id: search_tokens(text) for query_id, text in search_queries.items()}
morphology_queries = {query_id: morphology_tokens(text) for query_id, text in search_queries.items()}
normalized_plus_morph_queries = {
    query_id: [*normalized_queries[query_id], *morphology_queries[query_id]] for query_id in search_queries
}

chunk_pos_counts = Counter(token.split('_', 2)[1].upper() for tokens in morphology_documents.values() for token in tokens)
query_pos_counts = Counter(token.split('_', 2)[1].upper() for tokens in morphology_queries.values() for token in tokens)
length_ratio_by_level = {}
for level in ('card', 'page', 'section', 'benefit'):
    ratios = [len(normalized_plus_morph_documents[chunk['id']]) / max(1, len(normalized_documents[chunk['id']])) for chunk in chunks if chunk['metadata']['level'] == level]
    length_ratio_by_level[level] = {'count': len(ratios), 'median': percentile(ratios, 50), 'p95': percentile(ratios, 95)}
print({'kiwipiepy': kiwipiepy.__version__, 'kiwipiepy_model': importlib.metadata.version('kiwipiepy_model'), 'chunk_pos_counts': dict(chunk_pos_counts), 'query_pos_counts': dict(query_pos_counts), 'length_ratio_by_level': length_ratio_by_level})

{'kiwipiepy': '0.23.2', 'kiwipiepy_model': '0.23.0', 'chunk_pos_counts': {'NNP': 1406, 'NNG': 37890, 'VA': 535, 'VV': 1010}, 'query_pos_counts': {'NNG': 122, 'NNP': 10, 'VA': 2, 'VV': 1}, 'length_ratio_by_level': {'card': {'count': 10, 'median': 1.142291307025248, 'p95': 1.1651488711772762}, 'page': {'count': 50, 'median': 1.1334334415584415, 'p95': 1.1841437632135308}, 'section': {'count': 158, 'median': 1.1326647569499295, 'p95': 1.1872571819425446}, 'benefit': {'count': 109, 'median': 1.1417910447761195, 'p95': 1.1944544493039477}}}


## BM25 and cached-vector/RRF rankings

BM25는 `k1=1.5`, `b=0.75`, 동점은 chunk ID lexical 순이다. Vector는 cache 357개만 검증·사용한다. SQLite의 collection config가 `{}`이고 distance override metadata가 없음을 확인한 뒤 squared L2 NumPy full rank를 먼저 계산한다. Published top-5가 30/30이 아니면 snapshot Chroma의 approximate rank depth 50만 사용한다. RRF는 `k=60`, `0.5/0.5`, depth 50이다.

In [4]:
def bm25_scores(query_tokens, documents):
    tokenized = {identifier: list(tokens) for identifier, tokens in documents.items()}
    document_frequency = Counter(token for tokens in tokenized.values() for token in set(tokens))
    average_length = sum(map(len, tokenized.values())) / len(tokenized) if tokenized else 1.0
    scores = {}
    for identifier, tokens in tokenized.items():
        frequencies, score = Counter(tokens), 0.0
        for token in query_tokens:
            frequency = frequencies[token]
            if frequency:
                inverse_frequency = math.log(1 + (len(tokenized) - document_frequency[token] + 0.5) / (document_frequency[token] + 0.5))
                score += inverse_frequency * frequency * (BM25_K1 + 1) / (frequency + BM25_K1 * (1 - BM25_B + BM25_B * len(tokens) / average_length))
        scores[identifier] = score
    return scores


def rank_scores(scores):
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0]))]


normalized_bm25 = {
    query_id: rank_scores(bm25_scores(normalized_queries[query_id], normalized_documents)) for query_id in search_queries
}
normalized_plus_morph_bm25 = {
    query_id: rank_scores(bm25_scores(normalized_plus_morph_queries[query_id], normalized_plus_morph_documents)) for query_id in search_queries
}
published_normalized = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in saved_rows if row['configuration'] == 'normalized_bm25'}
assert all(normalized_bm25[query_id][:5] == published_normalized[query_id] for query_id in search_queries)

cached_items = [(f"chunk:{chunk['id']}", chunk['document']) for chunk in chunks] + [(f'query:{query_id}', text) for query_id, text in search_queries.items()]
assert len(cached_items) == embedding_usage['embedding_items'] == 357
vectors_by_key = {}
cache_validation = []
for batch_index, batch_start in enumerate(range(0, len(cached_items), 64)):
    batch = cached_items[batch_start:batch_start + 64]
    hashes = [hashlib.sha256(text.encode()).hexdigest() for _, text in batch]
    fingerprint = hashlib.sha256(canonical_json({'model': EMBEDDING_MODEL, 'hashes': hashes}).encode()).hexdigest()
    assert fingerprint == cache_fingerprints[batch_index]
    with np.load(CACHE_FILES[f'embedding_cache_{batch_index + 1}'], allow_pickle=False) as cached:
        embeddings = cached['embeddings']
        assert cached['hashes'].tolist() == hashes
        assert embeddings.shape == (len(batch), 1536)
        assert embeddings.dtype == np.float32 and np.isfinite(embeddings).all()
        vectors_by_key.update({key: vector.copy() for (key, _), vector in zip(batch, embeddings)})
        cache_validation.append({'batch': batch_index + 1, 'fingerprint': fingerprint, 'items': len(batch), 'dimension': 1536, 'dtype': str(embeddings.dtype), 'finite': True})
assert len(vectors_by_key) == 357

chunk_ids = [chunk['id'] for chunk in chunks]
chunk_vectors = np.stack([vectors_by_key[f'chunk:{identifier}'] for identifier in chunk_ids])
query_vectors = {query_id: vectors_by_key[f'query:{query_id}'] for query_id in search_queries}
published_vector = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in baseline_rows if row['method'] == 'vector'}


def l2_rank(query_vector):
    distances = np.sum((chunk_vectors - query_vector) ** 2, axis=1)
    return [identifier for identifier, _ in sorted(zip(chunk_ids, distances.tolist()), key=lambda item: (item[1], item[0]))]


numpy_vector_rank = {query_id: l2_rank(query_vectors[query_id]) for query_id in search_queries}
numpy_published_top5_exact_queries = sum(numpy_vector_rank[query_id][:5] == published_vector[query_id] for query_id in search_queries)
vector_fallback_used = numpy_published_top5_exact_queries != 30
distance_matches_squared_l2 = True

if vector_fallback_used:
    chroma_snapshot = tempfile.TemporaryDirectory()
    snapshot_root = Path(chroma_snapshot.name) / 'chroma'
    shutil.copytree(CHROMA_ROOT, snapshot_root)
    chroma_client = chromadb.PersistentClient(path=str(snapshot_root), settings=Settings(anonymized_telemetry=False))
    collection = chroma_client.get_collection(collection_name)
    vector_rank = {}
    for query_id in search_queries:
        response = collection.query(query_embeddings=[query_vectors[query_id].tolist()], n_results=RRF_DEPTH, include=['distances'])
        vector_rank[query_id] = response['ids'][0]
        by_id_l2 = {identifier: float(np.sum((chunk_vectors[index] - query_vectors[query_id]) ** 2)) for index, identifier in enumerate(chunk_ids)}
        distance_matches_squared_l2 = distance_matches_squared_l2 and np.allclose(response['distances'][0], [by_id_l2[identifier] for identifier in vector_rank[query_id]], rtol=2e-5, atol=2e-5)
    assert collection.count() == 327
    del collection, chroma_client
    gc.collect()
    chroma_snapshot.cleanup()
else:
    vector_rank = {query_id: ranking[:RRF_DEPTH] for query_id, ranking in numpy_vector_rank.items()}

published_vector_top5_exact_queries = sum(vector_rank[query_id][:5] == published_vector[query_id] for query_id in search_queries)
assert published_vector_top5_exact_queries == 30 and distance_matches_squared_l2


def rrf(keyword_ranking, vector_ranking):
    scores = defaultdict(float)
    for weight, ranking in zip(RRF_WEIGHTS, (keyword_ranking[:RRF_DEPTH], vector_ranking[:RRF_DEPTH])):
        for rank, identifier in enumerate(ranking, 1):
            scores[identifier] += weight / (RRF_K + rank)
    return rank_scores(scores)


rrf_vector_normalized = {query_id: rrf(normalized_bm25[query_id], vector_rank[query_id]) for query_id in search_queries}
rrf_vector_normalized_plus_morph = {query_id: rrf(normalized_plus_morph_bm25[query_id], vector_rank[query_id]) for query_id in search_queries}
assert rrf_vector_normalized == {query_id: rrf(normalized_bm25[query_id], vector_rank[query_id]) for query_id in search_queries}
assert normalized_plus_morph_bm25 == {query_id: rank_scores(bm25_scores(normalized_plus_morph_queries[query_id], normalized_plus_morph_documents)) for query_id in search_queries}
RANKINGS = {
    'normalized_bm25': normalized_bm25,
    'normalized_plus_morph_bm25': normalized_plus_morph_bm25,
    'rrf_vector_normalized': rrf_vector_normalized,
    'rrf_vector_normalized_plus_morph': rrf_vector_normalized_plus_morph,
}
print({'numpy_l2_published_top5_exact_queries': numpy_published_top5_exact_queries, 'fallback_to_tmp_snapshot': vector_fallback_used, 'snapshot_published_top5_exact_queries': published_vector_top5_exact_queries, 'snapshot_distances_match_squared_l2': distance_matches_squared_l2, 'cache_items': len(vectors_by_key), 'shape': chunk_vectors.shape, 'dtype': str(chunk_vectors.dtype), 'finite': bool(np.isfinite(chunk_vectors).all())})

{'numpy_l2_published_top5_exact_queries': 20, 'fallback_to_tmp_snapshot': True, 'snapshot_published_top5_exact_queries': 30, 'snapshot_distances_match_squared_l2': True, 'cache_items': 357, 'shape': (327, 1536), 'dtype': 'float32', 'finite': True}


## Evaluation, fixed decision, and artifacts

Ranking을 모두 만든 뒤에만 expected card/level/required terms/category를 relevance·metric·grouping에 사용한다. 결과는 4 configurations × 30 queries와 configuration별 6개 group으로 저장한다. Changed-query 감사에는 query 평문 대신 ID/category와 morph token overlap, top-5 변화, metric delta, 수기 분류를 남긴다.

In [5]:
def relevant_ids(evaluation):
    return {
        chunk['id'] for chunk in chunks
        if chunk['metadata']['card_key'] == evaluation['expected_card']
        and chunk['metadata']['level'] == evaluation['expected_level']
        and all(normalized_text(term) in normalized_text(chunk['document']) for term in evaluation['required_terms'])
    }


def metrics(evaluation, ranking):
    relevant = relevant_ids(evaluation)
    hits = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {
        'card_hit_at_3': int(any(chunk_by_id[identifier]['metadata']['card_key'] == evaluation['expected_card'] for identifier in ranking[:3])),
        'strict_evidence_hit_at_3': int(any(hits[:3])),
        'recall_at_5': sum(hits) / len(relevant),
        'mrr_at_5': 1 / first if first else 0.0,
        'ndcg_at_5': dcg / ideal if ideal else 0.0,
    }


per_query_rows = []
for configuration, rankings in RANKINGS.items():
    for query_id, evaluation in evaluation_by_id.items():
        ranking = rankings[query_id]
        per_query_rows.append({
            'configuration': configuration,
            'query_id': query_id,
            'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence',
            'category': evaluation['category'],
            'query': evaluation['query'],
            'expected_card': evaluation['expected_card'],
            'expected_level': evaluation['expected_level'],
            **metrics(evaluation, ranking),
            'top5_chunk_ids': canonical_json(ranking[:5]),
            'top5_cards': canonical_json([chunk_by_id[identifier]['metadata']['card_key'] for identifier in ranking[:5]]),
            'top5_levels': canonical_json([chunk_by_id[identifier]['metadata']['level'] for identifier in ranking[:5]]),
        })


def aggregate(rows):
    metric_names = ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5')
    return {**{metric: sum(row[metric] for row in rows) / len(rows) for metric in metric_names}, 'denominator': len(rows)}


summary_rows = []
for configuration in CONFIGURATION_NAMES:
    selected = [row for row in per_query_rows if row['configuration'] == configuration]
    groups = {
        'all': selected,
        'card': [row for row in selected if row['question_group'] == 'card'],
        'evidence': [row for row in selected if row['question_group'] == 'evidence'],
        **{f'category_{category}': [row for row in selected if row['category'] == category] for category in ('proper_noun', 'numeric_condition', 'semantic')},
    }
    for group, rows in groups.items():
        summary_rows.append({'configuration': configuration, 'question_group': group, **aggregate(rows)})

assert len(per_query_rows) == 4 * 30 == 120 and len(summary_rows) == 4 * 6 == 24
computed_rows = {(row['configuration'], row['query_id']): row for row in per_query_rows}
saved_row_map = {(row['configuration'], row['query_id']): row for row in saved_rows}
for configuration in ('normalized_bm25', 'rrf_vector_normalized'):
    for query_id in search_queries:
        current, previous = computed_rows[(configuration, query_id)], saved_row_map[(configuration, query_id)]
        for field in ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5', 'top5_chunk_ids', 'top5_cards', 'top5_levels'):
            assert str(current[field]) == previous[field], (configuration, query_id, field, current[field], previous[field])
computed_summary = {(row['configuration'], row['question_group']): row for row in summary_rows}
saved_summary_map = {(row['configuration'], row['question_group']): row for row in saved_summary['summaries']}
for configuration in ('normalized_bm25', 'rrf_vector_normalized'):
    for group in ('all', 'card', 'evidence', 'category_proper_noun', 'category_numeric_condition', 'category_semantic'):
        current, previous = computed_summary[(configuration, group)], saved_summary_map[(configuration, group)]
        assert current == previous, (configuration, group, current, previous)

rrf_base_evidence = computed_summary[('rrf_vector_normalized', 'evidence')]
rrf_morph_evidence = computed_summary[('rrf_vector_normalized_plus_morph', 'evidence')]
rrf_base_card = computed_summary[('rrf_vector_normalized', 'card')]
rrf_morph_card = computed_summary[('rrf_vector_normalized_plus_morph', 'card')]
mrr_delta = rrf_morph_evidence['mrr_at_5'] - rrf_base_evidence['mrr_at_5']
primary_pass = mrr_delta >= PROMOTION_RULE['primary']['minimum_delta']
guardrail_results = {
    'evidence_strict_evidence_hit_at_3': rrf_morph_evidence['strict_evidence_hit_at_3'] >= rrf_base_evidence['strict_evidence_hit_at_3'],
    'evidence_recall_at_5': rrf_morph_evidence['recall_at_5'] >= rrf_base_evidence['recall_at_5'],
    'evidence_ndcg_at_5': rrf_morph_evidence['ndcg_at_5'] >= rrf_base_evidence['ndcg_at_5'],
    'evidence_card_hit_at_3': rrf_morph_evidence['card_hit_at_3'] >= rrf_base_evidence['card_hit_at_3'],
    'card_card_hit_at_3': rrf_morph_card['card_hit_at_3'] >= rrf_base_card['card_hit_at_3'],
}
promotion_decision = 'promote_morphology_to_holdout_candidate' if primary_pass and all(guardrail_results.values()) else 'do_not_promote_morphology'
if 0 < mrr_delta < PROMOTION_RULE['primary']['minimum_delta']:
    decision_interpretation = 'inconclusive_non_promotion'
elif mrr_delta <= 0:
    decision_interpretation = 'non_promotion_no_positive_primary_delta'
elif not all(guardrail_results.values()):
    decision_interpretation = 'non_promotion_guardrail_failure'
else:
    decision_interpretation = 'development_candidate_only'

metric_names = ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5')
AUDIT_NOTES = {
    'bc_name': {'classification': 'beneficial_with_generic_false_positive', 'note': '법인/카드사/상품 일치로 card chunk가 RRF 2위에서 1위로 올라 MRR이 개선됐다. BM25에서는 같은 일반 명사를 가진 IBK 안내 page도 유입되어 false-positive 가능성은 남는다.'},
    'bc_numeric': {'classification': 'false_positive_common_terms_no_metric_change', 'note': '국내외/가맹점/적립 같은 공통 명사가 삼성·현대 chunk 순서를 바꿨지만 relevance metric은 변하지 않았다.'},
    'bc_semantic': {'classification': 'false_positive_common_terms_no_metric_change', 'note': '적립/포인트/결제 같은 공통 형태소로 우리카드 마일리지 section이 RRF top-5에 들어왔지만 기존 정답 section hit는 유지됐다.'},
    'nh_name': {'classification': 'within_card_level_shift_bm25_loss_rrf_stable', 'note': '나무/카드 같은 반복 형태소가 상세 benefit/page를 card overview보다 높여 BM25 MRR이 하락했다. RRF metric은 유지됐다.'},
    'nh_numeric': {'classification': 'metric_loss_repeated_term_frequency_bias', 'note': '스마트/캐시백/업종/적립이 같은 카드의 비정답 section에도 반복되어 정답 benefit이 RRF 3위에서 4위로 밀렸다. Strict Hit@3, MRR, nDCG가 하락했다.'},
    'hana_numeric': {'classification': 'within_card_reorder_no_loss', 'note': '소호/운영/경비/청구/할인 일치로 동일 카드 top-5 내부 순서만 바뀌었고 metric 손실은 없었다.'},
    'hana_semantic': {'classification': 'false_positive_common_terms_no_metric_change', 'note': '해외/결제/할인/혜택이 다른 카드에도 흔해 삼성 page가 롯데 page를 대체했다. 이미 정답 evidence가 없어 metric은 그대로였다.'},
    'hyundai_name': {'classification': 'over_segmentation_generic_token_no_metric_change', 'note': '발급사가 발급+사로 분석되어 morph_nng_사가 추가됐다. 이 짧은 일반 토큰과 카드가 타사 연회비/발급 chunk를 움직였지만 RRF metric은 변하지 않았다.'},
    'hyundai_numeric': {'classification': 'false_positive_common_fee_terms_no_metric_change', 'note': '본인/카드/연회비가 타사 연회비 chunk에도 일치해 top-5 tail이 교체됐지만 현대 정답 hit는 유지됐다.'},
    'hyundai_semantic': {'classification': 'ocr_typo_sensitive_bm25_loss_rrf_stable', 'note': '정답 계열 본문에 추가 혜택이 추가 헤택으로 OCR 오기되어 morph_nng_혜택 일치를 얻지 못했다. BM25 정답 순위가 한 단계 내려 MRR이 하락했지만 RRF metric은 유지됐다.'},
    'ibk_name': {'classification': 'false_positive_generic_product_terms_no_metric_change', 'note': '포인트/신용/카드/상품 같은 공통어로 우리카드 안내 section이 BM25 top-5에 유입됐지만 RRF 정답 metric은 유지됐다.'},
    'kb_name': {'classification': 'beneficial_compound_match', 'note': '프랜드카드 compound와 국민 형태소가 card overview를 RRF 2위에서 1위로 올려 MRR과 nDCG가 개선됐다.'},
    'lotte_name': {'classification': 'metric_loss_missing_distinctive_morph_and_generic_false_positive', 'note': '영문 상품명은 선택 POS 형태소가 되지 않고 query에는 카드사/상품만 남았다. 반복되는 일반 형태소가 compact card/name chunk를 top-5 밖으로 밀어 Recall 1.0, MRR 0.25, nDCG 0.431이 하락했다.'},
    'shinhan_numeric': {'classification': 'equivalent_evidence_swap_no_metric_change', 'note': '월납/공과금/할인이 같은 카드의 overview section 대신 benefit chunk를 올렸고 정답 hit metric은 동일했다.'},
    'shinhan_semantic': {'classification': 'over_segmentation_numeric_classifier_no_metric_change', 'note': '4대가 대라는 짧은 일반 명사로 분석되어 과분석 신호가 생겼지만 RRF top-5와 metric에는 실질 영향이 없었다.'},
    'woori_name': {'classification': 'bm25_benefit_rrf_stable', 'note': '카드사/상품 형태소가 card overview를 BM25 3위에서 1위로 올렸지만 RRF에서는 이미 1위여서 metric 변화는 없었다.'},
    'woori_numeric': {'classification': 'metric_loss_relevant_chunk_reorder', 'note': '기본/마일리지/적립/기준 반복으로 두 번째 정답 benefit이 RRF 3위에서 4위로 내려 nDCG가 0.0425 하락했다. Recall과 MRR은 유지됐다.'},
    'woori_semantic': {'classification': 'within_card_reorder_no_loss', 'note': '전월/실적/해외/결제/마일리지/혜택이 동일 카드 benefit 순서를 바꿨으나 정답 hit metric은 변하지 않았다.'},
}
changed_rows = []
for query_id, evaluation in evaluation_by_id.items():
    bm25_base = computed_rows[('normalized_bm25', query_id)]
    bm25_morph = computed_rows[('normalized_plus_morph_bm25', query_id)]
    rrf_base = computed_rows[('rrf_vector_normalized', query_id)]
    rrf_morph = computed_rows[('rrf_vector_normalized_plus_morph', query_id)]
    if bm25_base['top5_chunk_ids'] == bm25_morph['top5_chunk_ids'] and rrf_base['top5_chunk_ids'] == rrf_morph['top5_chunk_ids']:
        continue
    query_morph = morphology_queries[query_id]
    baseline_ids = json.loads(rrf_base['top5_chunk_ids'])
    morph_ids = json.loads(rrf_morph['top5_chunk_ids'])
    baseline_overlap = [token for token in query_morph if any(token in morphology_documents[identifier] for identifier in baseline_ids)]
    morph_overlap = [token for token in query_morph if any(token in morphology_documents[identifier] for identifier in morph_ids)]
    audit = AUDIT_NOTES.get(query_id, {'classification': 'manual_audit_pending', 'note': 'manual audit pending after first execution'})
    changed_rows.append({
        'query_id': query_id,
        'category': evaluation['category'],
        'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence',
        'bm25_baseline_top5_ids': bm25_base['top5_chunk_ids'],
        'bm25_morph_top5_ids': bm25_morph['top5_chunk_ids'],
        'bm25_baseline_top5_cards': bm25_base['top5_cards'],
        'bm25_morph_top5_cards': bm25_morph['top5_cards'],
        'bm25_baseline_top5_levels': bm25_base['top5_levels'],
        'bm25_morph_top5_levels': bm25_morph['top5_levels'],
        'rrf_baseline_top5_ids': rrf_base['top5_chunk_ids'],
        'rrf_morph_top5_ids': rrf_morph['top5_chunk_ids'],
        'rrf_baseline_top5_cards': rrf_base['top5_cards'],
        'rrf_morph_top5_cards': rrf_morph['top5_cards'],
        'rrf_baseline_top5_levels': rrf_base['top5_levels'],
        'rrf_morph_top5_levels': rrf_morph['top5_levels'],
        **{f'bm25_delta_{metric}': bm25_morph[metric] - bm25_base[metric] for metric in metric_names},
        **{f'rrf_delta_{metric}': rrf_morph[metric] - rrf_base[metric] for metric in metric_names},
        'query_morph_tokens': canonical_json(query_morph),
        'rrf_baseline_query_morph_overlap': canonical_json(baseline_overlap),
        'rrf_morph_query_morph_overlap': canonical_json(morph_overlap),
        'manual_audit_classification': audit['classification'],
        'manual_audit_note': audit['note'],
    })

atomic_csv(OUTPUT_ROOT / 'morphology_ablation_per_query.csv', per_query_rows)
atomic_csv(OUTPUT_ROOT / 'morphology_ablation_summary.csv', summary_rows)
atomic_csv(OUTPUT_ROOT / 'morphology_changed_queries.csv', changed_rows)

bm25_evidence = {
    'baseline': computed_summary[('normalized_bm25', 'evidence')],
    'morphology': computed_summary[('normalized_plus_morph_bm25', 'evidence')],
}
readme = f'''# Korean morphology retrieval ablation

이 디렉터리는 기존 개발 질의 30개로 수행한 morphology retrieval ablation 결과다. 독립 holdout은 사용하지 않았고, 이 결과만으로 운영 또는 미관측 데이터 일반화를 결론내릴 수 없다.

## Reproduction

- 환경: `conda run -n skn25`
- 도구: `nbclient 0.10.4`, `NotebookClient(timeout=900, kernel_name="python3")`
- 노트북: `notebooks/15_korean_morphology_retrieval_ablation.ipynb`
- network/API calls: 0; 기존 embedding cache만 사용

## Interpretation

- decision: `{promotion_decision}` (`{decision_interpretation}`)
- evidence RRF MRR@5 delta: `{mrr_delta:+.6f}` (사전 기준 `>= +0.025`)
- changed queries: `{len(changed_rows)}` / 30
- 설정은 개발셋 후보 비교용이며 형태소 적용의 보편적 우월성을 뜻하지 않는다.
'''
(OUTPUT_ROOT / 'README.md').write_text(readme, encoding='utf-8')

output_hashes = {
    name: sha256_file(OUTPUT_ROOT / name)
    for name in ('morphology_ablation_per_query.csv', 'morphology_ablation_summary.csv', 'morphology_changed_queries.csv', 'README.md')
}
summary_result = {
    'schema_version': 'korean_morphology_retrieval_ablation_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'scope': {'dataset': 'development_queries_only', 'queries': 30, 'independent_holdout_used': False, 'operational_generalization_claimed': False},
    'configurations': list(CONFIGURATION_NAMES),
    'retrieval_config': {'bm25': {'k1': BM25_K1, 'b': BM25_B, 'tie_break': 'chunk_id_lexical'}, 'rrf': {'k': RRF_K, 'weights': list(RRF_WEIGHTS), 'depth': RRF_DEPTH}},
    'morphology_config': {
        'package': 'kiwipiepy', 'package_version': kiwipiepy.__version__, 'model_package': 'kiwipiepy_model',
        'model_package_version': importlib.metadata.version('kiwipiepy_model'), 'num_workers': 1,
        'model': 'bundled default model', 'dictionary': 'default dictionary', 'user_dictionary': False,
        'synonyms': False, 'typo_correction': False, 'pos': list(POS_TAGS), 'form': 'analyzer form NFKC/lower',
        'namespace': 'morph_<tag_lower>_<form>', 'insertion': 'append one token per analyzer occurrence after normalized tokens; no dedup/reweight/double insertion',
        'applied_to': ['query', 'chunk document body'], 'excluded_from_ranking': ['metadata', 'structured_metadata', 'evaluation fields'],
    },
    'token_stats': {'chunk_pos_counts': dict(chunk_pos_counts), 'query_pos_counts': dict(query_pos_counts), 'length_ratio_by_level': length_ratio_by_level},
    'vector_reproduction': {
        'embedding_model': EMBEDDING_MODEL, 'cached_items': len(vectors_by_key), 'chunk_items': len(chunks), 'query_items': len(search_queries),
        'dimension': 1536, 'dtype': str(chunk_vectors.dtype), 'finite': bool(np.isfinite(chunk_vectors).all()),
        'cache_validation': cache_validation, 'sqlite_read_only': True, 'sqlite_collection_config': collection_config,
        'sqlite_distance_override_rows': distance_override_rows,
        'distance_basis': 'No explicit SQLite override; snapshot-returned distances empirically match NumPy squared L2.',
        'numpy_bruteforce_attempted_first': True, 'numpy_published_top5_exact_queries': numpy_published_top5_exact_queries,
        'fallback_to_tmp_byte_snapshot': vector_fallback_used, 'snapshot_published_top5_exact_queries': published_vector_top5_exact_queries,
        'snapshot_distances_match_squared_l2': bool(distance_matches_squared_l2),
        'rrf_vector_normalized_top5_and_summary_exact': True,
    },
    'baseline_reproduction': {'normalized_bm25_top5_and_summary_exact': True, 'rrf_vector_normalized_top5_and_summary_exact': True, 'queries': 30},
    'results': {'summaries': summary_rows, 'bm25_evidence': bm25_evidence, 'rrf_evidence_baseline': rrf_base_evidence, 'rrf_evidence_morphology': rrf_morph_evidence, 'rrf_evidence_mrr_delta': mrr_delta},
    'promotion': {'rule': PROMOTION_RULE, 'primary_pass': primary_pass, 'guardrail_results': guardrail_results, 'decision': promotion_decision, 'interpretation': decision_interpretation},
    'audit': {
        'changed_queries': len(changed_rows),
        'changed_query_ids': [row['query_id'] for row in changed_rows],
        'metric_loss_queries': [row['query_id'] for row in changed_rows if any(float(row[f'rrf_delta_{metric}']) < 0 for metric in metric_names)],
        'manual_audit_complete': all(row['manual_audit_classification'] != 'manual_audit_pending' for row in changed_rows),
        'observed_failure_modes': {
            'false_positive_common_terms': ['bc_numeric', 'bc_semantic', 'hana_semantic', 'ibk_name', 'lotte_name'],
            'over_segmentation': ['hyundai_name', 'shinhan_semantic'],
            'ocr_typo_sensitivity': ['hyundai_semantic'],
            'rrf_metric_losses': ['nh_numeric', 'lotte_name', 'woori_numeric'],
        },
    },
    'execution': {'environment': 'skn25', 'runner': 'nbclient 0.10.4 NotebookClient', 'network_calls': 0, 'api_calls': 0, 'new_embeddings': 0, 'package_install_calls': 0},
    'integrity': {'input_hashes_before': source_hashes_before, 'chroma_tree_hash_before': chroma_tree_hash_before, 'chroma_count_before': chroma_embedding_count_before, 'output_hashes': output_hashes},
    'limitations': [
        'Only 30 development queries were used; no independent generalization estimate is available.',
        'All configurations reuse structured-assisted chunk boundaries even though morphology reads body text only.',
        'NumPy exact L2 did not reproduce every approximate Chroma top-5, so RRF uses a byte-copied temporary snapshot rank at depth 50.',
        'Kiwi analysis can over-segment OCR noise or attach common content morphemes to irrelevant chunks.',
        'BC selected excerpt and IBK incomplete/ambiguous source coverage limitations remain inherited from the input artifacts.',
    ],
}
atomic_json(OUTPUT_ROOT / 'morphology_ablation_summary.json', summary_result)

manifest_files = {
    **{f'input:{name}': {'path': path.relative_to(PROJECT_ROOT).as_posix(), 'sha256': source_hashes_before[name]} for name, path in {**SOURCE_FILES, **CACHE_FILES}.items()},
    **{f'output:{name}': {'path': f'notebooks/data/15_korean_morphology_retrieval_ablation/{name}', 'sha256': sha256_file(OUTPUT_ROOT / name)} for name in ('morphology_ablation_per_query.csv', 'morphology_ablation_summary.csv', 'morphology_ablation_summary.json', 'morphology_changed_queries.csv', 'README.md')},
    'output:notebook': {'path': NOTEBOOK_PATH.relative_to(PROJECT_ROOT).as_posix(), 'sha256': 'pending_after_nbclient_serialization'},
}
atomic_json(OUTPUT_ROOT / 'morphology_run_manifest.json', {'schema_version': 'korean_morphology_run_manifest_v1', 'self_hash_excluded': True, 'files': manifest_files})
print({'bm25_evidence': bm25_evidence, 'rrf_evidence_baseline': rrf_base_evidence, 'rrf_evidence_morphology': rrf_morph_evidence, 'rrf_evidence_mrr_delta': mrr_delta, 'decision': promotion_decision, 'interpretation': decision_interpretation, 'guardrails': guardrail_results, 'changed_queries': len(changed_rows), 'manual_audit_complete': summary_result['audit']['manual_audit_complete']})

{'bm25_evidence': {'baseline': {'configuration': 'normalized_bm25', 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.7, 'recall_at_5': 0.6875, 'mrr_at_5': 0.49749999999999994, 'ndcg_at_5': 0.52214939569159, 'denominator': 20}, 'morphology': {'configuration': 'normalized_plus_morph_bm25', 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.7, 'recall_at_5': 0.6875, 'mrr_at_5': 0.48083333333333333, 'ndcg_at_5': 0.5141214746064088, 'denominator': 20}}, 'rrf_evidence_baseline': {'configuration': 'rrf_vector_normalized', 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.75, 'recall_at_5': 0.7125, 'mrr_at_5': 0.6033333333333333, 'ndcg_at_5': 0.5965505415086586, 'denominator': 20}, 'rrf_evidence_morphology': {'configuration': 'rrf_vector_normalized_plus_morph', 'question_group': 'evidence', 'card_hit_at_3': 0.9, 'strict_evidence_hit_at_3': 0.7, 'recall_at_5': 0.7125, 'mrr_at_5': 0.5991666666666667, 

## Integrity checks

결과 파일을 다시 읽어 schema/count를 확인하고, 허용된 원본 입력·cache·Chroma의 실행 전후 raw SHA-256/count 동일성을 검증한다. Manifest는 self hash를 제외하며, 실행 드라이버가 최종 serialized notebook hash를 채운다.

In [6]:
source_hashes_after = {name: sha256_file(path) for name, path in {**SOURCE_FILES, **CACHE_FILES}.items()}
chroma_tree_hash_after = tree_hash(CHROMA_ROOT)
with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
    chroma_embedding_count_after = connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
assert source_hashes_after == source_hashes_before
assert chroma_tree_hash_after == chroma_tree_hash_before
assert chroma_embedding_count_after == chroma_embedding_count_before == 327

final_summary = json.loads((OUTPUT_ROOT / 'morphology_ablation_summary.json').read_text(encoding='utf-8'))
final_summary['integrity'].update({
    'input_hashes_after': source_hashes_after,
    'chroma_tree_hash_after': chroma_tree_hash_after,
    'chroma_count_after': chroma_embedding_count_after,
    'inputs_cache_chroma_unchanged': True,
})
atomic_json(OUTPUT_ROOT / 'morphology_ablation_summary.json', final_summary)
final_manifest = json.loads((OUTPUT_ROOT / 'morphology_run_manifest.json').read_text(encoding='utf-8'))
final_manifest['files']['output:morphology_ablation_summary.json']['sha256'] = sha256_file(OUTPUT_ROOT / 'morphology_ablation_summary.json')
atomic_json(OUTPUT_ROOT / 'morphology_run_manifest.json', final_manifest)

stored_per_query = list(csv.DictReader((OUTPUT_ROOT / 'morphology_ablation_per_query.csv').open(encoding='utf-8')))
stored_summary = list(csv.DictReader((OUTPUT_ROOT / 'morphology_ablation_summary.csv').open(encoding='utf-8')))
stored_changed = list(csv.DictReader((OUTPUT_ROOT / 'morphology_changed_queries.csv').open(encoding='utf-8')))
stored_json = json.loads((OUTPUT_ROOT / 'morphology_ablation_summary.json').read_text(encoding='utf-8'))
assert len(stored_per_query) == 120 and len(stored_summary) == 24 and len(stored_changed) == len(changed_rows)
assert {row['configuration'] for row in stored_per_query} == set(CONFIGURATION_NAMES)
assert all(sum(row['configuration'] == configuration for row in stored_per_query) == 30 for configuration in CONFIGURATION_NAMES)
assert stored_json['baseline_reproduction']['normalized_bm25_top5_and_summary_exact']
assert stored_json['baseline_reproduction']['rrf_vector_normalized_top5_and_summary_exact']
assert stored_json['vector_reproduction']['snapshot_published_top5_exact_queries'] == 30
assert stored_json['execution']['network_calls'] == stored_json['execution']['api_calls'] == 0
assert stored_json['audit']['manual_audit_complete']
assert not any(row['manual_audit_classification'] == 'manual_audit_pending' for row in stored_changed)
assert json.loads((OUTPUT_ROOT / 'morphology_run_manifest.json').read_text(encoding='utf-8'))['self_hash_excluded']
print({'code_cells_compile': 'validated externally and all cells executed', 'per_query_rows': len(stored_per_query), 'summary_rows': len(stored_summary), 'changed_rows': len(stored_changed), 'source_hashes_unchanged': True, 'chroma_tree_hash_unchanged': True, 'chroma_count_unchanged': True, 'network_api_calls': 0})

{'code_cells_compile': 'validated externally and all cells executed', 'per_query_rows': 120, 'summary_rows': 24, 'changed_rows': 18, 'source_hashes_unchanged': True, 'chroma_tree_hash_unchanged': True, 'chroma_count_unchanged': True, 'network_api_calls': 0}
